# LangSmith Client Anonymizer — PII Redaction in LangGraph

This notebook tests the PII redaction setup in `agent.py`. All agent code, anonymizer patterns, and client configuration are imported directly from `agent.py` — nothing is redefined here.

---

### How it works

The `Client(anonymizer=...)` intercepts all data being sent to LangSmith and applies regex patterns before export. 

### What this notebook covers

1. Confirm the anonymizer patterns work before touching the graph
2. Run and verify the LSD pattern via `compile_agent`
3. Run and verify the non-LSD pattern via `tracing_context`

## 1. Setup

Load environment variables and confirm `LANGSMITH_ENDPOINT` is pointing at the right instance. If this prints `NOT SET`, traces will go to LangSmith SaaS (`smith.langchain.com`). For a self-hosted instance, set `LANGSMITH_ENDPOINT` in your `.env` file.

The `Client` in `agent.py` reads `LANGSMITH_ENDPOINT` at the moment it is instantiated. If `load_dotenv()` hasn't run yet at that point, the `Client` silently defaults to `smith.langchain.com`. The `langsmith_client.api_url` check below confirms what the `Client` actually resolved to — both values should match.

In [36]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_PROJECT", "pii-masking-langgraph")

print("LANGSMITH_ENDPOINT:", os.environ.get("LANGSMITH_ENDPOINT", "NOT SET — defaulting to smith.langchain.com"))

LANGSMITH_ENDPOINT: https://api.smith.langchain.com


In [37]:
from agent import langsmith_client

# Confirm the Client in agent.py resolved to the correct endpoint
# If this doesn't match LANGSMITH_ENDPOINT above, Client was initialized before load_dotenv() ran
print("Client api_url:     ", langsmith_client.api_url)

Client api_url:      https://api.smith.langchain.com


## 2. Verify the Anonymizer in Isolation

Run this before touching the graph. It tells you immediately whether the patterns in `agent.py` are working.

- **If this redacts correctly but the trace still shows PII** — the wiring is broken (`tracing_context` missing or `langsmith_client` doesn't have the anonymizer attached)
- **If this doesn't redact** — the patterns are broken, fix them in `agent.py` before moving on

In [38]:
from agent import anonymizer

test = "My SSN is 123-45-6789, email is user@example.com, phone (555) 867-5309, card 4532-1488-0343-6728"
print(anonymizer(test))

My SSN is [SSN_REDACTED], email is [EMAIL_REDACTED], phone [PHONE_REDACTED], card [CC_REDACTED]


## 3. Run the Agent — LSD Pattern

In production with LSD, `compile_agent` is the entry point. `langgraph.json` points at it:

```json
{ "graphs": { "my_agent": "./langgraph-example/agent.py:compile_agent" } }
```

`compile_agent` is an async context manager that wraps the compiled graph in `tracing_context` for its entire lifetime. LSD calls it once at startup — every request flows through it automatically.

To test locally, import and use it exactly as LSD does. If redaction works here, it will work in deployment.

In [39]:
from agent import compile_agent
from langchain_core.messages import HumanMessage

user_message = HumanMessage(content="My SSN is 123-45-6789 and my email is user@example.com. Can you confirm you received this?")

async with compile_agent() as compiled:
    result = compiled.invoke({"messages": [user_message]})

print(result["messages"][-1].content)

I'm sorry, but I can't process or store personal information such as Social Security numbers or email addresses. If you have any other questions or need assistance, feel free to ask!


## 4. Verify the LSD Trace

Fetch the most recent trace and check every span for raw PII. The anonymizer should have replaced all values before they reached LangSmith.

The `reader_client` is a plain `Client()` with no anonymizer — it reads back exactly what LangSmith stored, confirming redaction happened at the source.

In [40]:
import time
import json
from langsmith import Client

time.sleep(5)

reader_client = Client()
project_name = os.environ.get("LANGSMITH_PROJECT", "pii-masking-langgraph")

runs = list(reader_client.list_runs(
    project_name=project_name,
    is_root=True,
    limit=1,
))

if not runs:
    print("No traces found. Check your LANGSMITH_API_KEY and project name.")
else:
    root_run = runs[0]
    print(f"Trace ID:  {root_run.id}")
    print(f"Trace URL: {root_run.url}\n")

Trace ID:  019c971a-0a11-74f2-bfec-7616b7d8baaf
Trace URL: https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/projects/p/63359a66-ae1c-4e8a-a06b-d6b5e6a16549/r/019c971a-0a11-74f2-bfec-7616b7d8baaf?trace_id=019c971a-0a11-74f2-bfec-7616b7d8baaf&start_time=2026-02-25T23:19:52.081903



In [41]:
spans = list(reader_client.list_runs(
    trace_id=root_run.trace_id,
    project_name=project_name,
))

raw_pii = {
    "ssn":   "123-45-6789",
    "email": "user@example.com",
}

print(f"Checking {len(spans)} spans for raw PII...\n")

leaks_found = []

for span in spans:
    traced = json.dumps({"inputs": span.inputs, "outputs": span.outputs})
    for pii_type, pii_value in raw_pii.items():
        if pii_value in traced:
            leaks_found.append((span.name, pii_type, pii_value))

if leaks_found:
    print("WARNING — PII LEAK DETECTED:")
    for span_name, pii_type, pii_value in leaks_found:
        print(f"   Span '{span_name}' contains raw {pii_type}: {pii_value}")
    print("\nCheck that langgraph.json points at compile_agent and that langsmith_client has the anonymizer attached.")
else:
    print("ALL CLEAN — no raw PII found in any traced span.")
    print("\nWhat LangSmith stored for the root input:")
    print(json.dumps(root_run.inputs, indent=2))

Checking 3 spans for raw PII...

ALL CLEAN — no raw PII found in any traced span.

What LangSmith stored for the root input:
{
  "messages": [
    {
      "additional_kwargs": {},
      "content": "My SSN is [SSN_REDACTED] and my email is [EMAIL_REDACTED]. Can you confirm you received this?",
      "id": "5d9c0be2-41e9-4788-9065-6e80cfd2920c",
      "response_metadata": {},
      "type": "human"
    }
  ]
}


## 5. Run the Agent — Non-LSD Pattern

For scripts not running through LSD, wrap each `agent.invoke()` call in `tracing_context`. The anonymizer is active for the duration of the `with` block only — any call outside this block sends unredacted traces to LangSmith with no error or warning.

Note: `agent` and `langsmith_client` are imported directly from `agent.py` — no rebuilding needed.

In [42]:
from agent import agent, langsmith_client
from langchain_core.messages import HumanMessage
import langsmith as ls

user_message = HumanMessage(content="My SSN is 123-45-6789 and my email is user@example.com. Can you confirm you received this?")

with ls.tracing_context(client=langsmith_client):
    result = agent.invoke({"messages": [user_message]})

print(result["messages"][-1].content)

I'm sorry, but I can't process or store personal information such as Social Security numbers or email addresses. If you have any other questions or need assistance, feel free to ask!


## 6. Verify the Non-LSD Trace

Same verification as Section 4.

In [43]:
time.sleep(5)

runs = list(reader_client.list_runs(
    project_name=project_name,
    is_root=True,
    limit=1,
))

if not runs:
    print("No traces found. Check your LANGSMITH_API_KEY and project name.")
else:
    root_run = runs[0]
    print(f"Trace ID:  {root_run.id}")
    print(f"Trace URL: {root_run.url}\n")

Trace ID:  019c971a-adfa-73d1-95ac-40c92fb00247
Trace URL: https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/projects/p/63359a66-ae1c-4e8a-a06b-d6b5e6a16549/r/019c971a-adfa-73d1-95ac-40c92fb00247?trace_id=019c971a-adfa-73d1-95ac-40c92fb00247&start_time=2026-02-25T23:20:34.042892



In [44]:
spans = list(reader_client.list_runs(
    trace_id=root_run.trace_id,
    project_name=project_name,
))

print(f"Checking {len(spans)} spans for raw PII...\n")

leaks_found = []

for span in spans:
    traced = json.dumps({"inputs": span.inputs, "outputs": span.outputs})
    for pii_type, pii_value in raw_pii.items():
        if pii_value in traced:
            leaks_found.append((span.name, pii_type, pii_value))

if leaks_found:
    print("WARNING — PII LEAK DETECTED:")
    for span_name, pii_type, pii_value in leaks_found:
        print(f"   Span '{span_name}' contains raw {pii_type}: {pii_value}")
    print("\nCheck that tracing_context is wrapping agent.invoke() and that langsmith_client has the anonymizer attached.")
else:
    print("ALL CLEAN — no raw PII found in any traced span.")
    print("\nWhat LangSmith stored for the root input:")
    print(json.dumps(root_run.inputs, indent=2))

Checking 3 spans for raw PII...

ALL CLEAN — no raw PII found in any traced span.

What LangSmith stored for the root input:
{
  "messages": [
    {
      "additional_kwargs": {},
      "content": "My SSN is [SSN_REDACTED] and my email is [EMAIL_REDACTED]. Can you confirm you received this?",
      "id": "1def9c84-b54e-4b81-b7d2-a3d07d68950a",
      "response_metadata": {},
      "type": "human"
    }
  ]
}
